In [1]:
from collections import defaultdict
import csv
from fitting_GUI import *

In [2]:
def load_table(path):
    table = defaultdict(list) #element -> [(line, energy, intensity), ...]
    with open(path, newline="") as fh:
        for r in csv.DictReader(fh):
            table[r["element"]].append((r["line"], float(r["energy"]), float(r["intensity"])))
    return table

def weighted_energy(lines, prefix):
    """Intensity-weighted mean energy of all lines starting with 'Ka' or 'Kb'."""
    pts = [(E, I) for name, E, I in lines if name.startswith(prefix)]
    return sum(E * I for E, I in pts) / sum(I for _, I in pts)

def identify(r, table, top=3):
    """r = results[folder][key]. Compares the two fitted peaks with each element."""
    peaks = [r["peak1"], r["peak2"]]
    if None in peaks:
        print("Need two successful fits")
        return
    a, b = sorted(peaks, key=lambda p: p["mu"]) #lower energy = Kalpha, higher = Kbeta
    sig_a, sig_b = a["fwhm"] / 2.3548, b["fwhm"] / 2.3548

    scores = []
    for el, lines in table.items():
        Ea, Eb = weighted_energy(lines, "Ka"), weighted_energy(lines, "Kb")
        chi2 = ((a["mu"] - Ea) / sig_a) ** 2 + ((b["mu"] - Eb) / sig_b) ** 2
        scores.append((chi2, el, Ea, Eb))

    print(f"measured: K\u03b1 {a['mu']:.2f} keV, K\u03b2 {b['mu']:.2f} keV")
    for chi2, el, Ea, Eb in sorted(scores)[:top]:
        print(f"  {el:>3}: K\u03b1 {Ea:.2f}, K\u03b2 {Eb:.2f}  (score {chi2:.2f})")

In [3]:
results = load_results()

table = load_table("xray_energies.csv")

for folder, d in results.items():
    if folder == "calibration":
        continue
    for key, r in d.items():
        print(f"\n{folder}/{key}")
        identify(r, table)


measurements/week2_metal_1
Need two successful fits

measurements/week2_metal_2
Need two successful fits

measurements/week2_metal_3
measured: Kα 78.29 keV, Kβ 82.91 keV
   Pb: Kα 74.16, Kβ 85.27  (score 0.24)
   Bi: Kα 76.25, Kβ 87.69  (score 0.28)
   Tl: Kα 72.11, Kβ 82.90  (score 0.41)

measurements/week3_metal_1
measured: Kα 58.21 keV, Kβ 65.08 keV
   Ta: Kα 57.07, Kβ 65.48  (score 0.25)
    W: Kα 58.83, Kβ 67.51  (score 0.71)
   Re: Kα 60.62, Kβ 69.58  (score 3.26)

measurements/week3_metal_2
measured: Kα 66.04 keV, Kβ 74.57 keV
   Pt: Kα 66.20, Kβ 76.05  (score 0.24)
   Ir: Kα 64.30, Kβ 73.85  (score 0.54)
   Au: Kα 68.13, Kβ 78.29  (score 2.19)

measurements/week3_metal_3
measured: Kα 71.31 keV, Kβ 80.64 keV
   Hg: Kα 70.10, Kβ 80.57  (score 0.22)
   Tl: Kα 72.11, Kβ 82.90  (score 0.57)
   Au: Kα 68.13, Kβ 78.29  (score 2.06)
